# Data Preprocessing

## Import Libraries

In [201]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from imblearn.under_sampling import RandomUnderSampler
import scipy.stats

## Read the Data

In [202]:
client_train = pd.read_csv('data/client_train.csv', low_memory=False)
invoice_train = pd.read_csv('data/invoice_train.csv', low_memory=False)

client_test = pd.read_csv(f'data/client_test.csv', low_memory=False)
invoice_test = pd.read_csv(f'data/invoice_test.csv', low_memory=False)
sample_submission = pd.read_csv(f'results/SampleSubmission.csv', low_memory=False)

In [203]:
client_train.head()

,disrict,client_id,client_catg,region,creation_date,target
0,60,train_Client_0,11,101,31/12/1994,0.0
1,69,train_Client_1,11,107,29/05/2002,0.0
2,62,train_Client_10,11,301,13/03/1986,0.0
3,69,train_Client_100,11,105,11/07/1996,0.0
4,62,train_Client_1000,11,303,14/10/2014,0.0


## Data Cleaning

In [204]:
train = invoice_train.copy()
# counter_statue == 269375 -> can be droped, since there is just one observation and also the train_Client_53725 belong to only that observation 
train = train.query("counter_statue != 269375")
# counter_statue == 420 and reading_remarque == 5 and counter_code == 1 -> they are just one time in the same observation, so we can drop them as well
train = train.query("counter_statue != 420")
# replace string of int to int
train['counter_statue'] = train['counter_statue'].replace('0', 0)
train['counter_statue'] = train['counter_statue'].replace('1', 1)
train['counter_statue'] = train['counter_statue'].replace('4', 4)
train['counter_statue'] = train['counter_statue'].replace('5', 5)
# counter_statue == 769 and reading_remarque == 207 ???
train = train.query("counter_statue != 769")
# counter_statue == 618 and reading_remarque == 413 ???
train = train.query("counter_statue != 618")
# counter_statue == 46 and reading_remarque == 203 ???
train = train.query("counter_statue != 46")
#months_number > 88
train = train.query("months_number <= 88") # auch aus testdaten
invoice_train = train.copy()

invoice_test = invoice_test.query("months_number <= 88")

## Train Test Split

In [205]:
# split client train, stratify by target variable, and set random state for reproducibility
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(client_train.drop(columns=['target']), client_train['target'], test_size=0.2, stratify=client_train['target'], random_state=42)

In [206]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)

(108394, 5) (108394,)
(27099, 5) (27099,)


In [207]:
client_train = pd.merge(X_train, y_train, left_index=True, right_index=True)
client_val = pd.merge(X_val, y_val, left_index=True, right_index=True)

### undersampling

In [208]:
# undersample the majority class in the training data
rus = RandomUnderSampler(random_state=42)
X_train_res, y_train_res = rus.fit_resample(X_train, y_train)

In [209]:
clients_in_resampled_train = X_train_res['client_id']
clients_in_val = X_val['client_id']

invoice_train_res = invoice_train[invoice_train['client_id'].isin(clients_in_resampled_train)]
invoice_val = invoice_train[invoice_train['client_id'].isin(clients_in_val)]

client_train_res = pd.merge(X_train_res, y_train_res, left_index=True, right_index=True)

In [210]:
print(X_train_res.shape, y_train_res.shape)
print(X_val.shape, y_val.shape)

(12106, 5) (12106,)
(27099, 5) (27099,)


## Feature Engineering Functions

In [211]:
#convert the column invoice_date to date time format on both the invoice train and invoice test
def invoice_convert_date(df):
    df_updated = df.copy()
    df_updated['invoice_date'] = pd.to_datetime(df_updated['invoice_date'])
    return df_updated

In [212]:
# convert tarif_type, counter_statue, counter_code, reading_remarque, counter_type to categorical data type
def invoice_to_category(df):
    df_updated = df.copy()
    df_updated['tarif_type'] = df_updated['tarif_type'].astype('category')
    df_updated['counter_statue'] = df_updated['counter_statue'].astype('category')
    df_updated['counter_code'] = df_updated['counter_code'].astype('category')
    df_updated['reading_remarque'] = df_updated['reading_remarque'].astype('category')
    df_updated['counter_type'] = df_updated['counter_type'].astype('category')
    return df_updated

In [213]:
# create variable that sums the consumption for all 4 levels
def invoice_create_consumption(df):
    df_updated = df.copy()
    df_updated['total_consumption'] = df_updated['new_index'] - df_updated['old_index']
    df_updated['consumption_per_month'] = df_updated['total_consumption'] / df_updated['months_number']
    return df_updated

In [214]:
# convert disrict, client_catg, region to categorical data type
def client_to_category(df):
    df_updated = df.copy()
    df_updated['disrict'] = df_updated['disrict'].astype('category')
    df_updated['client_catg'] = df_updated['client_catg'].astype('category')
    df_updated['region'] = df_updated['region'].astype('category')
    return df_updated

In [215]:
# convert creation_date to date time format on both the client train and client test
def client_convert_date(df):
    df_updated = df.copy()
    df_updated['creation_date'] = pd.to_datetime(df_updated['creation_date'], dayfirst=True)
    # create new variable with 7 bins for the creation date using cut
    df_updated['creation_date_bin'] = pd.cut(df_updated['creation_date'], bins=7, labels=False)
    # convert the creation_date_bin to categorical data type
    df_updated['creation_date_bin'] = df_updated['creation_date_bin'].astype('category')
    return df_updated

In [216]:
# DEFINE LEVEL USAGE
# -------------------------
def define_activity_level4_usage(df):
    df_updated = df.copy()
    df_updated['used_lvl4'] = (df_updated['consommation_level_4'] > 0).astype(int)
    # activity
    # any consumption at all (levels 1–4)
    df_updated['active'] = (
        (df_updated['consommation_level_1'] > 0) |
        (df_updated['consommation_level_2'] > 0) |
        (df_updated['consommation_level_3'] > 0) |
        (df_updated['consommation_level_4'] > 0)
    ).astype(int)
    return df_updated

In [217]:
# -------------------------
# FRACTION PER CLIENT
# -------------------------
def lvl4_fraction(df):
    df_updated = df.copy()
    client_lvl4_fraction = df_updated.groupby('client_id')['used_lvl4'].mean().reset_index()
    client_lvl4_fraction.rename(columns={
        'used_lvl4': 'frac_time_lvl4'
        }, inplace=True)
    df_updated['frac_time_lvl4'] = df_updated['client_id'].map(
        client_lvl4_fraction.set_index('client_id')['frac_time_lvl4']
    )
    return df_updated

In [218]:
def duplicate_invoices_same_day(df):
    df = df.copy()
    grp = df.groupby(['client_id', 'counter_type', 'invoice_date'])['client_id']
    # count rows per (client_id, counter_type, invoice_date)
    per_day_count = grp.transform('size')
    # per-row boolean: this row belongs to a duplicated same-day group
    duplicated_group = per_day_count.gt(1)
    # per-client flag: any duplicated same-day group for that client
    df['has_multi_invoice_same_day'] = duplicated_group.groupby(df['client_id']).transform('any').astype('int8')
    return df

In [219]:
# date gaps
def date_gaps(df):
    df_updated = df.copy()
    df_updated = df_updated.sort_values(['client_id', 'invoice_date'])
    df_updated['gap_days'] = df_updated.groupby(['client_id', 'counter_type'])['invoice_date'].diff().dt.days
    return df_updated

In [220]:
def invoice_scaled_consumption_per_month(df):
    df_updated = df.copy()
    df_updated['consumption_per_month_scaled'] = df_updated.groupby(['client_id', 'counter_type'])['consumption_per_month'].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.5)
    return df_updated

In [221]:
def correct_newIndex(df):
    updated_df = df.copy()
    updated_df['index_diff'] = updated_df['new_index'] - updated_df['old_index']
    updated_df.loc[updated_df['index_diff'] < 0, 'new_index'] += 100000
    return updated_df

### Run on (undersampled) data

In [223]:
invoice_train_res = invoice_convert_date(invoice_train_res)
invoice_train_res = correct_newIndex(invoice_train_res)
invoice_train_res = invoice_to_category(invoice_train_res)
invoice_train_res = invoice_create_consumption(invoice_train_res)
invoice_train_res = define_activity_level4_usage(invoice_train_res)
invoice_train_res = lvl4_fraction(invoice_train_res)
invoice_train_res = duplicate_invoices_same_day(invoice_train_res)
invoice_train_res = date_gaps(invoice_train_res)
invoice_train_res = invoice_scaled_consumption_per_month(invoice_train_res)
# write the preprocessed invoice train_res to csv
invoice_train_res.to_csv('data/created_invoice_train_res.csv', index=False)

In [224]:
invoice_val = invoice_convert_date(invoice_val)
invoice_val = correct_newIndex(invoice_val)
invoice_val = invoice_to_category(invoice_val)
invoice_val = invoice_create_consumption(invoice_val)
invoice_val = define_activity_level4_usage(invoice_val)
invoice_val = lvl4_fraction(invoice_val)
invoice_val = duplicate_invoices_same_day(invoice_val)
invoice_val = date_gaps(invoice_val)
invoice_val = invoice_scaled_consumption_per_month(invoice_val)
# write the preprocessed invoice val to csv
invoice_val.to_csv('data/created_invoice_val.csv', index=False)

In [225]:
invoice_test = correct_newIndex(invoice_test)
invoice_test = invoice_convert_date(invoice_test)
invoice_test = invoice_to_category(invoice_test)
invoice_test = invoice_create_consumption(invoice_test)
invoice_test = define_activity_level4_usage(invoice_test)
invoice_test = lvl4_fraction(invoice_test)
invoice_test = duplicate_invoices_same_day(invoice_test)
invoice_test = date_gaps(invoice_test)
invoice_test = invoice_scaled_consumption_per_month(invoice_test)
# write the preprocessed invoice test to csv
invoice_test.to_csv('data/created_invoice_test.csv', index=False)

In [226]:
for df in [client_train_res, client_val, client_test]:
    df = client_convert_date(df)
    df = client_to_category(df)

client_train_res.to_csv('data/created_client_train_res.csv', index=False)
client_val.to_csv('data/created_client_val.csv', index=False)
client_test.to_csv('data/created_client_test.csv', index=False)

In [ ]:
# read created train_res data
invoice_train_res = pd.read_csv('data/created_invoice_train_res.csv')
# read created data
invoice_val = pd.read_csv('data/created_invoice_val.csv')
# read created data
invoice_test = pd.read_csv('data/created_invoice_test.csv')

# read created client data
client_train_res = pd.read_csv('data/created_client_train_res.csv')
client_val = pd.read_csv('data/created_client_val.csv')
client_test = pd.read_csv('data/created_client_test.csv')

In [228]:
invoice_train_res.tail()

,client_id,invoice_date,tarif_type,counter_number,counter_statue,counter_code,reading_remarque,counter_coefficient,consommation_level_1,consommation_level_2,...,counter_type,index_diff,total_consumption,consumption_per_month,used_lvl4,active,frac_time_lvl4,has_multi_invoice_same_day,gap_days,consumption_per_month_scaled
474955,train_Client_99992,2017-10-19,11,48616,0,207,9,1,1124,0,...,ELEC,1124,1124,140.50,0,1,0.0,0,244.0,0.393657
474956,train_Client_99992,2018-02-19,11,48616,0,207,9,1,562,0,...,ELEC,562,562,140.50,0,1,0.0,0,123.0,0.393657
474957,train_Client_99992,2018-10-22,11,48616,0,207,9,1,517,0,...,ELEC,517,517,129.25,0,1,0.0,0,245.0,0.351679
474958,train_Client_99992,2019-02-20,11,48616,0,207,8,1,454,0,...,ELEC,454,454,113.50,0,1,0.0,0,121.0,0.292910
474959,train_Client_99992,2019-06-24,11,48616,0,207,8,1,490,0,...,ELEC,490,490,122.50,0,1,0.0,0,124.0,0.326493


## Aggregate functions on invoice to client

In [229]:
def aggregate_by_client_id(invoice_data):
    aggs = {}
    aggs['gap_days'] = [np.nanmean]
    aggs['gap_days'] = [np.nanstd]
    aggs['consumption_per_month'] = [np.nanmean]
    aggs['consumption_per_month'] = [np.nanstd]
    aggs['frac_time_lvl4'] = [np.nanmax]
    aggs['consumption_per_month_scaled'] = [np.nanstd]
    aggs['active'] = ['sum']
    aggs['active'] = [np.nanmean]
    aggs['counter_statue'] = [np.nanstd]
    aggs['reading_remarque'] = [np.nanstd]
    aggs['tarif_type'] = [scipy.stats.mode]
    aggs['index_diff'] = [np.nanstd]
    aggs['has_multi_invoice_same_day'] = [np.nanmax]
    aggs['used_lvl4'] = ['sum']


    agg_trans = invoice_data.groupby(['client_id']).agg(aggs)
    agg_trans.columns = ['_'.join(col).strip() for col in agg_trans.columns.values]
    agg_trans.reset_index(inplace=True)

    df = (invoice_data.groupby('client_id')
            .size()
            .reset_index(name='{}transactions_count'.format('1')))
    return pd.merge(df, agg_trans, on='client_id', how='left')

In [230]:
#group invoice data by client_id
agg_train_res = aggregate_by_client_id(invoice_train_res)
agg_val = aggregate_by_client_id(invoice_val)
agg_test = aggregate_by_client_id(invoice_test)

In [231]:
agg_train_res.head()

,client_id,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,train_Client_100032,18,49.380203,32.538439,0.0,0.185140,0.944444,0.383482,0.970143,"(11, 18)",177.558018,0,0
1,train_Client_10004,2,NaN,156.093822,0.0,0.707107,1.000000,0.000000,0.000000,"(11, 2)",624.375288,0,0
2,train_Client_100070,6,9.237604,25.636725,0.0,0.484004,0.833333,0.000000,1.643168,"(11, 3)",102.546900,0,0
3,train_Client_100083,56,117.524205,84.745520,0.0,0.266665,0.910714,0.000000,1.360744,"(40, 30)",445.423317,0,0
4,train_Client_100091,43,40.106976,51.578769,0.0,0.318639,0.744186,0.152499,1.395293,"(11, 24)",218.892769,0,0


In [232]:
# replace tarif_type_mode by first entry of that column
agg_train_res['tarif_type_mode'] = agg_train_res['tarif_type_mode'].apply(lambda x: x[0] if isinstance(x, tuple) else x)
# change to category type
agg_train_res['tarif_type_mode'] = agg_train_res['tarif_type_mode'].astype('category')

# same for val and test
agg_val['tarif_type_mode'] = agg_val['tarif_type_mode'].apply(lambda x: x[0] if isinstance(x, tuple) else x)
agg_val['tarif_type_mode'] = agg_val['tarif_type_mode'].astype('category')
agg_test['tarif_type_mode'] = agg_test['tarif_type_mode'].apply(lambda x: x[0] if isinstance(x, tuple) else x)
agg_test['tarif_type_mode'] = agg_test['tarif_type_mode'].astype('category')

In [233]:
#merge aggregate data with client dataset
train_res = pd.merge(client_train_res, agg_train_res, on='client_id', how='left')
display(train_res)

train_res.to_csv('data/created_train_res.csv', index=False)

#same for val 
val = pd.merge(client_val, agg_val, on='client_id', how='left')
display(val)

val.to_csv('data/created_val.csv', index=False)

# merge client test with agg test
test = pd.merge(client_test, agg_test, on='client_id', how='left')
display(test)

test.to_csv('data/created_test.csv', index=False)

,disrict,client_id,client_catg,region,creation_date,target,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,62,train_Client_75200,11,304,28/03/2002,0.0,42.0,77.133721,36.430692,0.000000,0.226629,0.952381,0.000000,1.380341,11,145.722767,0.0,0.0
1,62,train_Client_76567,11,301,29/11/2007,0.0,66.0,18.534154,9.978589,0.000000,0.204394,0.424242,0.123091,1.158731,40,23.391629,0.0,0.0
2,63,train_Client_97383,11,311,05/08/1991,0.0,38.0,73.577052,88.665034,0.000000,0.230000,1.000000,0.000000,1.289249,11,367.966455,0.0,0.0
3,60,train_Client_104767,11,101,20/09/1994,0.0,34.0,58.449781,25.916130,0.000000,0.317989,1.000000,0.000000,1.142284,10,133.201802,0.0,0.0
4,60,train_Client_84837,11,101,17/11/1994,0.0,85.0,70.541808,122.868681,0.000000,0.238324,1.000000,0.000000,1.355557,11,481.619241,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12101,63,train_Client_117834,11,313,18/04/1981,1.0,35.0,91.618839,108.238709,0.000000,0.194762,0.971429,0.000000,1.367080,11,445.511126,0.0,0.0
12102,60,train_Client_101401,11,101,19/04/1977,1.0,71.0,161.948353,446.370612,0.056338,0.259272,0.690141,0.000000,1.150278,11,992.942701,1.0,4.0
12103,63,train_Client_80409,11,312,14/06/2001,1.0,70.0,67.816884,68.827065,0.000000,0.203148,0.985714,0.000000,1.328459,11,364.073657,0.0,0.0
12104,69,train_Client_64228,11,103,05/06/2010,1.0,36.0,210.802659,109.254962,0.000000,0.310523,0.972222,0.000000,1.125110,11,1080.755766,0.0,0.0


,disrict,client_id,client_catg,region,creation_date,target,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,69,train_Client_15159,11,104,24/12/2008,0.0,26.0,71.863714,69.225383,0.000000,0.284293,0.923077,0.196116,1.336471,11,288.338380,0.0,0.0
1,63,train_Client_80740,11,311,13/09/2014,0.0,30.0,53.249537,49.349953,0.000000,0.291712,1.000000,0.000000,0.547723,11,169.041004,0.0,0.0
2,63,train_Client_59100,11,311,07/11/1989,1.0,55.0,119.886288,161.799264,0.054545,0.345812,0.763636,0.134840,1.361570,10,1003.806981,0.0,3.0
3,69,train_Client_125675,11,105,10/11/1993,0.0,58.0,135.559102,149.164036,0.000000,0.275661,0.982759,0.000000,1.308911,11,715.475605,0.0,0.0
4,63,train_Client_4885,11,313,29/12/2012,0.0,32.0,86.278352,36.327523,0.000000,0.258478,0.500000,0.296145,1.524002,11,144.914644,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27094,62,train_Client_106726,11,304,16/11/1998,0.0,43.0,37.224965,3.021254,0.000000,0.151063,0.976744,1.071115,1.096658,11,7.450910,1.0,0.0
27095,60,train_Client_83056,11,101,28/02/1992,0.0,26.0,173.623184,43.246750,0.000000,0.279011,0.923077,0.000000,1.250846,11,409.988099,0.0,0.0
27096,69,train_Client_92212,11,104,22/11/2008,0.0,31.0,98.590439,367.482570,0.516129,0.187563,1.000000,0.179605,1.469401,11,1465.477179,0.0,16.0
27097,63,train_Client_76137,11,312,28/03/2002,0.0,43.0,47.695413,32.841130,0.000000,0.247713,0.976744,0.762493,1.365206,11,131.364519,0.0,0.0


,disrict,client_id,client_catg,region,creation_date,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,62,test_Client_0,11,307,28/05/2002,37.0,44.900834,40.424028,0.000000,0.175757,0.972973,0.000000,1.221061,11,235.684859,0.0,0.0
1,69,test_Client_1,11,103,06/08/2009,22.0,117.157647,722.359649,0.045455,0.194222,1.000000,0.213201,1.216766,11,3200.849986,0.0,1.0
2,62,test_Client_10,11,310,07/04/2004,74.0,13.134901,105.694968,0.013514,0.241883,0.986486,0.000000,1.482216,11,422.779872,0.0,1.0
3,60,test_Client_100,11,101,08/10/1992,40.0,96.299793,66.060236,0.000000,0.293850,0.950000,0.000000,1.034966,11,247.253171,0.0,0.0
4,62,test_Client_1000,11,301,21/07/1977,53.0,113.376261,158.358003,0.000000,0.280807,0.924528,0.686803,1.319443,11,864.294814,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58064,63,test_Client_9995,11,399,17/03/2010,4.0,0.000000,133.188991,0.000000,0.408248,0.500000,0.000000,0.000000,11,532.755964,0.0,0.0
58065,63,test_Client_9996,11,311,28/05/2011,46.0,58.724011,23.653269,0.000000,0.220683,0.978261,0.759163,1.100066,11,107.556838,0.0,0.0
58066,60,test_Client_9997,11,101,04/03/1978,59.0,120.360325,31.634983,0.000000,0.220892,1.000000,0.000000,1.065094,10,116.987086,0.0,0.0
58067,60,test_Client_9998,11,101,23/02/2018,1.0,NaN,NaN,0.000000,NaN,1.000000,NaN,NaN,11,NaN,0.0,0.0


## Tips 
- Thorough EDA and incorporating domain knowledge
- Re-grouping categorical features
- More feature engineering(try utilizing some date-time features)
- Target balancing - oversampling, undersampling, SMOTE, scale_pos_weight
- Model ensembling
- Train-test split or cross-validation


# ******************* GOOD LUCK!!! ***************************